# Step 06 - LLM Based Response Generation

## Objective

In this notebook we implement a Large Language Model based
Natural Language Generation system.

Input:

Intent + Slots

Example:

intent = order_status
order_id = ORD123

Output:

I can help with order status for order ORD123.


## Why LLM?

The custom Decoder Transformer trained earlier learns patterns
from limited training data.

However, it struggles with:

- unseen slot values
- copying IDs
- generalization

LLMs are pretrained on massive text corpora and can perform
instruction following using prompts.

We evaluate whether LLM improves:

- Response quality
- Slot preservation
- Hallucination reduction

In [39]:
#CELL 2 - Imports

import json
import os
import time
from tqdm import tqdm

In [40]:
#CELL 3 - Load Validation Data

DATA_PATH = "../data/"


with open(
    DATA_PATH + "linearized_validation.json"
) as f:

    val_data = json.load(f)


print(
    "Validation Samples:",
    len(val_data)
)

Validation Samples: 60


In [41]:
# CELL 4 - Understand One Sample

sample = val_data[0]

sample

{'sample_id': 'SMP0154',
 'intent': 'order_status',
 'slots': {'order_id': 'ORD841324',
  'phone_last4': '9488',
  'channel': 'website'},
 'linearized_sequence': '[BOS] intent : order_status <sep> channel : website <sep> order_id : ORD841324 <sep> phone_last4 : 9488 [EOS]',
 'target_text': 'I can help with order status for order ORD841324.',
 'edge_case': False,
 'edge_case_types': []}

In [42]:
#CELL 5 - Prompt Builder

# def build_prompt(sample):


#     prompt = f"""

# You are a customer service response generation assistant.


# Task:
# Generate a natural language response using ONLY the provided intent and slots.


# Rules:

# 1. Preserve all IDs exactly.
# 2. Do not invent missing information.
# 3. Do not add extra details.
# 4. Generate one short customer support response.



# Input:

# {sample["linearized_sequence"]}



# Response:

# """


#     return prompt

# CELL 5 - Prompt Builder

# CELL 5 - Prompt Builder
# Controlled prompt creation for LLM based generation


def clean_input(text):

    """
    Convert decoder formatted sequence into
    LLM friendly structured input.

    Removes:
    - BOS/EOS tokens
    - unnecessary slots

    Keeps:
    - intent
    - important slot values
    """

    # Remove decoder special tokens

    text = text.replace("[BOS]", "")

    text = text.replace("[EOS]", "")


    # Convert separator into lines

    text = text.replace("<sep>", "\n")


    lines = text.split("\n")


    allowed_slots = [

        "intent",

        "order_id"

    ]


    filtered_lines = []


    for line in lines:

        line = line.strip()


        for slot in allowed_slots:

            if line.startswith(slot):

                filtered_lines.append(
                    line
                )


    return "\n".join(
        filtered_lines
    )




def build_prompt(sample):


    cleaned_input = clean_input(

        sample["linearized_sequence"]

    )


    prompt = f"""

You are a customer service response generator.

Your task:
Convert structured intent and slot information into a short customer response.


Strict Rules:

1. Generate only the final customer message.
2. Do not output slot names.
3. Do not include unnecessary fields.
4. Preserve order IDs exactly.
5. Do not add explanations.



Examples:


Input:
intent : order_status
order_id : ORD123


Output:
I can help with order status for order ORD123.



Input:
intent : refund_status
order_id : ORD456


Output:
I can help with refund status for order ORD456.



Input:
intent : modify_order
order_id : ORD789


Output:
I can help with modify order for order ORD789.



Input:
intent : product_availability


Output:
I can help with product availability. Please share the missing details for verification.



Now generate.


Input:

{cleaned_input}


Output:

"""


    return prompt


In [43]:
# Testing above prompt 
# 
print(
    build_prompt(
        val_data[0]
    )
)



You are a customer service response generator.

Your task:
Convert structured intent and slot information into a short customer response.


Strict Rules:

1. Generate only the final customer message.
2. Do not output slot names.
3. Do not include unnecessary fields.
4. Preserve order IDs exactly.
5. Do not add explanations.



Examples:


Input:
intent : order_status
order_id : ORD123


Output:
I can help with order status for order ORD123.



Input:
intent : refund_status
order_id : ORD456


Output:
I can help with refund status for order ORD456.



Input:
intent : modify_order
order_id : ORD789


Output:
I can help with modify order for order ORD789.



Input:
intent : product_availability


Output:
I can help with product availability. Please share the missing details for verification.



Now generate.


Input:

intent : order_status
order_id : ORD841324


Output:




In [44]:
#CELL 6 - Install LLM Library   

#For assignment, easiest is HuggingFace.

#In terminal:

#`pip install transformers accelerate sentencepiece`

#Then:

#`pip freeze > requirements.txt`

In [45]:
#CELL 7 - Load Small LLM

# from transformers import pipeline


# generator = pipeline(
#     task="text2text-generation",
#     model="google/flan-t5-small"
# )


# print("LLM Loaded Successfully")

# from transformers import (
#     AutoTokenizer,
#     AutoModelForSeq2SeqLM
# )


# MODEL_NAME = "google/flan-t5-small"


# tokenizer = AutoTokenizer.from_pretrained(
#     MODEL_NAME
# )


# llm_model = AutoModelForSeq2SeqLM.from_pretrained(
#     MODEL_NAME
# )


# print("LLM Loaded")

# CELL 7 - Load Small LLM

from transformers import pipeline


generator = pipeline(
    "text2text-generation",
    model="google/flan-t5-small"
)


print("LLM Loaded Successfully")

/Users/pavankumararepu/BitsPilani/2nd Year/AIMLCZG521_Con_AI/Assignment1/AssignmentSolution/venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


LLM Loaded Successfully


In [46]:
# CELL 8 - Generate Single Response 

prompt = build_prompt(
    val_data[0]
)

result = generator(
    prompt,
    max_new_tokens=50,
    do_sample=False,
    repetition_penalty=1.5
)

print("INPUT:")
print(val_data[0]["linearized_sequence"])

print("\nEXPECTED:")
print(val_data[0]["target_text"])

print("\nLLM GENERATED:")
print(result[0]["generated_text"])

INPUT:
[BOS] intent : order_status <sep> channel : website <sep> order_id : ORD841324 <sep> phone_last4 : 9488 [EOS]

EXPECTED:
I can help with order status for order ORD841324.

LLM GENERATED:
I want to order status for order ORD841324


In [47]:
#CELL 9 — Generate All Validation Responses

llm_predictions=[]


start=time.time()


for item in tqdm(val_data):


    prompt = build_prompt(
        item
    )


    result = generator(

        prompt,

        max_length=80,

        do_sample=False

    )


    llm_predictions.append(

        {
            "input":
            item["linearized_sequence"],


            "expected":
            item["target_text"],


            "generated":
            result[0]["generated_text"]

        }

    )


end=time.time()


print(
    "Total time:",
    end-start
)

100%|██████████| 60/60 [00:40<00:00,  1.46it/s]

Total time: 41.0218780040741


In [48]:
# CELL 10 - Save Output

os.makedirs(
    "../outputs",
    exist_ok=True
)


with open(

    "../outputs/llm_predictions.json",

    "w"

) as f:


    json.dump(

        llm_predictions,

        f,

        indent=4

    )


print(
    "Saved:",
    len(llm_predictions)
)

Saved: 60


In [49]:
# CELL 11 - Compare Examples

for i in range(5):


    print("="*60)


    print(
        "INPUT:"
    )

    print(
        llm_predictions[i]["input"]
    )


    print(
        "\nEXPECTED:"
    )

    print(
        llm_predictions[i]["expected"]
    )


    print(
        "\nLLM:"
    )

    print(
        llm_predictions[i]["generated"]
    )

INPUT:
[BOS] intent : order_status <sep> channel : website <sep> order_id : ORD841324 <sep> phone_last4 : 9488 [EOS]

EXPECTED:
I can help with order status for order ORD841324.

LLM:
I want to order status for order ORD841324
INPUT:
[BOS] intent : product_availability <sep> pincode : 110536 <sep> products : milk <sep> store_id : STR4656 [EOS]

EXPECTED:
I can help with product availability. Please share the missing details for verification.

LLM:
I want to buy a new product
INPUT:
[BOS] intent : modify_order <sep> delivery_slot : 8 PM - 10 PM <sep> order_id : ORD191208 <sep> products : shampoo | green tea <sep> quantity_change : increase to 2 packs [EOS]

EXPECTED:
I can help with modify order for order ORD191208.

LLM:
I am trying to modify order_id ORD191208
INPUT:
[BOS] intent : payment_issue <sep> issue_type : charged_but_order_not_placed <sep> order_id : ORD556870 <sep> payment_method : wallet <sep> transaction_id : TXN24984645 [EOS]

EXPECTED:
I can help with payment issue for o

# LLM Generation Observations

A pretrained FLAN-T5 model was used for slot conditioned response generation.

Initial observations:

- Direct decoder formatted inputs caused poor generation because BOS/EOS tokens are training artifacts.
- Input preprocessing improved LLM understanding.
- Few-shot prompting improved controlled generation.


Strengths:

- Better preservation of unseen slot values.
- No task specific model training required.
- Handles new IDs better than custom decoder.


Limitations:

- Smaller LLM models may produce grammar variations.
- Prompt quality strongly impacts output.
- Some intents require better examples.


Overall:

LLM based generation showed better generalization compared to the custom Transformer decoder.